In [1]:
# cell 1
# from google.cloud import storage
from datetime import datetime
from google.colab import files
from datetime import datetime
# from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig
import torch
# from torch.nn.functional import binary_cross_entropy_with_logits, binary_cross_entropy
from torch.nn import BCEWithLogitsLoss

# from torch.optim import AdamW
# from torch.utils.data import DataLoader
from tqdm import tqdm
import io
import pandas as pd
import numpy as np
import os
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import f1_score, accuracy_score

import re

from google.cloud import storage
from sklearn.model_selection import KFold
from transformers import get_linear_schedule_with_warmup
from torch.nn import init
import torch.nn as nn
import torch
from torch.utils.data import Dataset, DataLoader
# from transformers import RobertaTokenizer, RobertaForSequenceClassification, AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig, AdamW
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score
import numpy as np
import pandas as pd
from tqdm import tqdm
from google.cloud import storage
import json
import gc

In [2]:
# cell 2
def download_table_from_gcs(project_id, bucket_name, source_blob_name):
  storage_client = storage.Client(project=project_id)
  bucket = storage_client.bucket(bucket_name)
  blob = bucket.blob(source_blob_name)
  contents = blob.download_as_bytes()

  # Read the CSV data into a pandas DataFrame
  try:
      df = pd.read_csv(io.BytesIO(contents))
      # print(df.head()) #display first few rows to verify.
      return df
  except pd.errors.EmptyDataError:
      print("CSV file is empty.")
  except pd.errors.ParserError:
      print("Error parsing CSV file. Check its format.")
  except Exception as e:
      print(f"An error occurred: {e}")



def get_data(bucket_name, blob_folder, df):
  storage_client = storage.Client()
  bucket = storage_client.bucket(bucket_name)
  blobs = bucket.list_blobs(prefix=blob_folder)

  for i, blob in enumerate(blobs):
    file_name = blob.name.split('/')[-1]
    if file_name in df['path'].values:
      with blob.open("r") as f:
         df.loc[df['path'] == file_name, 'text'] = f.read()

# creating a dataset class and object for the training via pytorch api
# class MultiLabelDataset(torch.utils.data.Dataset):
#     def __init__(self, encodings, labels):
#         self.encodings = encodings
#         self.labels = labels
#         # self.encodings = torch.tensor(encodings)
#         # self.labels = torch.tensor(labels)
#         #

#     def __len__(self):
#         return len(self.labels)

#     def __getitem__(self, idx):
#         item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
#         item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
#         # item = {key: val[idx] for key, val in self.encodings.items()}
#         # item["labels"] = self.labels[idx], dtype=torch.float
#         return item

# class MultiLabelDataset2(torch.utils.data.Dataset):
#     def __init__(self, texts, labels, tokenizer,):
#         self.texts = texts
#         self.labels = labels
#         self.tokenizer = tokenizer

#     def __len__(self):
#         return len(self.texts)

#     def __getitem__(self, idx):
#         text = self.texts[idx]
#         label = self.labels[idx]

#         encoding = self.tokenizer(text, padding="max_length", truncation=True, max_length=512, return_tensors="pt")
#         input_ids = encoding['input_ids'].squeeze()
#         attention_mask = encoding['attention_mask'].squeeze()

#         return {
#             'input_ids': input_ids.clone().detach(),
#             'attention_mask': attention_mask.clone().detach(),
#             'label': torch.tensor(label, dtype=torch.float).clone().detach()
#         }

In [3]:
# cell 2.1
def download_model_from_gcs(bucket_name, source_folder, dest_folder):
    client = storage.Client()
    bucket = client.bucket(bucket_name)

    blobs = bucket.list_blobs(prefix=source_folder)
    os.makedirs(dest_folder, exist_ok=True)

    for blob in blobs:
        # Construct the relative destination path
        relative_path = os.path.relpath(blob.name, source_folder)
        dest_path = os.path.join(dest_folder, relative_path)

        # Check if blob is a file (not a directory)
        if not blob.name.endswith('/'):
            os.makedirs(os.path.dirname(dest_path), exist_ok=True)
            blob.download_to_filename(dest_path)
            print(f"Downloaded {blob.name} to {dest_path}")
        else:
            print(f"Skipped directory: {blob.name}")





In [4]:
# cell 3 - for pretrained
project_id = "iucc-social-media-influencers"
bucket_name = "influencers_tiktok"
source_blob_name = "tables/sample_labels_38_values_07_01_2025.csv"

model_folder_path = "models/pretrained/deberta-v2-xlarge-pretrained_31_12_2024"
input_model_path = "deberta-xlarge-pretrained-itamar-gcp" #
download_model_from_gcs(bucket_name, model_folder_path, input_model_path)
batch_size = 4 # 8 for roberta, 4 for deberta

date = datetime.now().strftime("%d_%m_%y")
run_id = f"deberta-pretrained_38val_{date}"

df = download_table_from_gcs(project_id, bucket_name, source_blob_name)
df[df.columns[2:]] = df[df.columns[2:]].astype(int)

Skipped directory: models/pretrained/deberta-v2-xlarge-pretrained_31_12_2024/
Downloaded models/pretrained/deberta-v2-xlarge-pretrained_31_12_2024/added_tokens.json to deberta-xlarge-pretrained-itamar-gcp/added_tokens.json
Downloaded models/pretrained/deberta-v2-xlarge-pretrained_31_12_2024/config.json to deberta-xlarge-pretrained-itamar-gcp/config.json
Downloaded models/pretrained/deberta-v2-xlarge-pretrained_31_12_2024/model.safetensors to deberta-xlarge-pretrained-itamar-gcp/model.safetensors
Downloaded models/pretrained/deberta-v2-xlarge-pretrained_31_12_2024/optimizer.pt to deberta-xlarge-pretrained-itamar-gcp/optimizer.pt
Downloaded models/pretrained/deberta-v2-xlarge-pretrained_31_12_2024/special_tokens_map.json to deberta-xlarge-pretrained-itamar-gcp/special_tokens_map.json
Downloaded models/pretrained/deberta-v2-xlarge-pretrained_31_12_2024/spm.model to deberta-xlarge-pretrained-itamar-gcp/spm.model
Downloaded models/pretrained/deberta-v2-xlarge-pretrained_31_12_2024/tokenizer

In [5]:
# cell 4
import numpy
def save_df(df, file_name):
    df.to_csv(file_name, index_label='Row Header')
    files.download(file_name)

df['label'] = df[df.columns[2:]].apply(pd.Series.tolist, axis=1)
# df['path'] = df['path'].apply(lambda x: "".join([x,".mp4"]))
df['text'] = ""

# print(type(df['label']))
# print(df['label'][:2])
# print(df['label'].shape)
# len(df['label'][0])#why is it 40 and not 38?
# print("First row raw data:", df['label'].iloc[0])

# Try converting using list() to ensure we have proper lists
# Method 1: Using pandas directly
df_new = pd.DataFrame(df['label'].tolist())
print("\nShape of new DataFrame:", df_new.shape)
print(df_new.head(10))
# save_df(df_new, "input38labels.csv")
# # Method 2: Using list comprehension to ensure clean lists
# #clean_lists = [list(row) if isinstance(row, (list, np.ndarray)) else row for row in df['label']]
# #df_new = pd.DataFrame(clean_lists)
# #print("Shape of new DataFrame:", df_new.shape)  # Should be (890, 40)




def calculate_column_sums(series):
    # Convert series of lists into a 2D numpy array
    # This creates an array of shape (890, 40)
    array_2d = np.vstack(series.values)

    # Calculate sum for each column
    # This gives us an array of shape (40,) containing the sums
    column_sums = array_2d.sum(axis=0)

    return column_sums
print(calculate_column_sums(df_new))


Shape of new DataFrame: (890, 38)
   0   1   2   3   4   5   6   7   8   9   ...  28  29  30  31  32  33  34  \
0   0   0   0   0   0   0   0   0   0   0  ...   0   1   0   0   0   0   0   
1   0   0   0   0   1   0   0   1   0   0  ...   0   0   0   0   1   0   1   
2   0   1   1   0   0   0   1   0   0   0  ...   0   0   0   0   0   0   0   
3   0   0   0   0   0   1   0   0   0   0  ...   0   0   1   0   0   0   0   
4   0   1   0   0   1   1   0   0   0   0  ...   0   0   1   0   0   0   0   
5   0   0   0   0   0   1   0   0   1   0  ...   0   0   0   0   0   0   0   
6   0   0   0   0   0   0   0   0   0   0  ...   0   0   1   0   0   0   0   
7   0   0   0   0   0   0   0   0   1   0  ...   0   0   0   0   1   0   0   
8   0   1   0   0   0   1   0   0   1   0  ...   0   0   0   0   0   0   0   
9   0   0   0   0   0   1   0   0   0   0  ...   0   0   1   0   0   0   0   

   35  36  37  
0   0   0   0  
1   0   0   0  
2   0   0   0  
3   0   0   0  
4   0   0   0  
5   0   0 

In [6]:
# cell 5
videos_folder = "movies/videos/VideoScripts"
get_data(bucket_name, videos_folder, df)

In [7]:
# cell 6
# ################################15:56 13/01: max_length=512;cross validation, take only texts >600 and split once for all labels
# from google.cloud import storage
# from sklearn.model_selection import KFold
# from transformers import get_linear_schedule_with_warmup
# from torch.nn import init
# import torch.nn as nn
# import torch
# from torch.utils.data import Dataset, DataLoader
# from transformers import RobertaTokenizer, RobertaForSequenceClassification, AdamW
# from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig
# from sklearn.metrics import classification_report, precision_score, recall_score, f1_score
# import numpy as np
# import pandas as pd
# from tqdm import tqdm
# from google.cloud import storage
# import json
# import gc

def calculate_metrics_per_label(label_name, true_labels, predicted_labels):
    """
    Calculate precision, recall, and F1 score for a given label
    """
    precision = precision_score(true_labels, predicted_labels, zero_division=0)
    recall = recall_score(true_labels, predicted_labels, zero_division=0)
    f1 = f1_score(true_labels, predicted_labels, zero_division=0)

    # Calculate true positives, false positives, true negatives, false negatives
    tp = np.sum((true_labels == 1) & (predicted_labels == 1))
    fp = np.sum((true_labels == 0) & (predicted_labels == 1))
    tn = np.sum((true_labels == 0) & (predicted_labels == 0))
    fn = np.sum((true_labels == 1) & (predicted_labels == 0))

    return {
        'label': label_name,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'tp': tp,
        'fp': fp,
        'tn': tn,
        'fn': fn,
        'support': np.sum(true_labels == 1)
    }

def save_to_gcp(data, project_id, bucket_name, blob_name):
    client = storage.Client(project=project_id)
    bucket = client.get_bucket(bucket_name)
    blob = bucket.blob(blob_name)
    blob.upload_from_string(data)

def filter_by_token_length(texts, labels, max_tokens=600):
    """Filter out texts longer than max_tokens"""
    # tokenizer = RobertaTokenizer.from_pretrained('roberta-large')
    tokenizer = AutoTokenizer.from_pretrained(input_model_path)
    filtered_texts = []
    filtered_labels = []

    for text, label in zip(texts, labels):
        tokens = tokenizer.encode(text, add_special_tokens=True)
        if len(tokens) <= max_tokens:
            filtered_texts.append(text)
            filtered_labels.append(label)

    return filtered_texts, filtered_labels

def save_split_to_gcp(train_indices, test_indices, project_id, bucket_name, blob_name):
    """Save the best split indices to GCP"""
    from google.cloud import storage

    split_data = {
        'train_indices': train_indices.tolist(),
        'test_indices': test_indices.tolist()
    }

    client = storage.Client(project=project_id)
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(blob_name)
    blob.upload_from_string(json.dumps(split_data))

def calculate_class_weights(labels):
    """Calculate class weights for imbalanced datasets"""
    total = len(labels)
    pos = sum(labels)
    neg = total - pos
    if pos == 0 or neg == 0:  # Handle edge case
        return torch.tensor([1.0, 1.0])
    weight_pos = total / (2 * pos)
    weight_neg = total / (2 * neg)
    return torch.tensor([weight_neg, weight_pos])

class EarlyStopping:
    def __init__(self, patience=3, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0
class BalancedBatchSampler(torch.utils.data.Sampler):
    """Sampler that ensures balanced batches"""
    def __init__(self, labels, batch_size):
        self.labels = labels
        self.batch_size = batch_size
        self.pos_indices = [i for i, label in enumerate(labels) if label == 1]
        self.neg_indices = [i for i, label in enumerate(labels) if label == 0]
        self.pos_batch_size = batch_size // 2
        self.neg_batch_size = batch_size - self.pos_batch_size

    def __iter__(self):
        pos_indices = np.random.permutation(self.pos_indices)
        neg_indices = np.random.permutation(self.neg_indices)
        pos_batches = [pos_indices[i:i + self.pos_batch_size]
                      for i in range(0, len(pos_indices), self.pos_batch_size)]
        neg_batches = [neg_indices[i:i + self.neg_batch_size]
                      for i in range(0, len(neg_indices), self.neg_batch_size)]

        # Use the shorter length to avoid index errors
        min_batches = min(len(pos_batches), len(neg_batches))
        indices = []
        for i in range(min_batches):
            indices.extend(pos_batches[i])
            indices.extend(neg_batches[i])

        return iter(indices)

    def __len__(self):
        return len(self.labels)

def train_epoch(model, data_loader, optimizer, device, scheduler=None, class_weights=None):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []

    for batch in tqdm(data_loader, desc="Training"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        # Use weighted loss
        if class_weights is not None:
            loss_fct = torch.nn.BCEWithLogitsLoss(pos_weight=class_weights[1].to(device))
        else:
            loss_fct = torch.nn.BCEWithLogitsLoss()

        loss = loss_fct(outputs.logits.view(-1), labels.float())

        # Store predictions and labels for monitoring
        with torch.no_grad():
            preds = torch.sigmoid(outputs.logits).cpu().numpy()
            all_preds.extend(preds.flatten())
            all_labels.extend(labels.cpu().numpy().flatten())

        total_loss += loss.item()
        loss.backward()

        # Gradient clipping to prevent explosion
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        if scheduler is not None:
            scheduler.step()

    # Print distribution of predictions
    preds_array = np.array(all_preds)
    print(f"\nPrediction statistics:")
    print(f"Mean: {preds_array.mean():.4f}")
    print(f"Std: {preds_array.std():.4f}")
    print(f"Min: {preds_array.min():.4f}")
    print(f"Max: {preds_array.max():.4f}")

    return total_loss / len(data_loader)

def evaluate_with_metrics(model, data_loader, device, threshold=0.5):
    model.eval()
    predictions = []
    actual_labels = []
    raw_predictions = []

    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels']

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.sigmoid(outputs.logits).cpu().numpy().flatten()
            raw_predictions.extend(probs)
            predictions.extend(probs >= threshold)
            actual_labels.extend(labels.cpu().numpy().flatten())

    # If all predictions are the same, try adjusting threshold
    if (len(set(predictions)) == 1) and (threshold == 0.5):
        print("\nAdjusting threshold due to uniform predictions...")
        raw_preds = np.array(raw_predictions)
        new_threshold = np.percentile(raw_preds, 75)  # Use 75th percentile as threshold
        threshold = new_threshold
        predictions = (raw_preds >= threshold).tolist()
        print(f"New threshold: {threshold:.4f}")
        print(f"New prediction distribution: {sum(predictions)} positive, {len(predictions)-sum(predictions)} negative")

    return np.array(actual_labels), np.array(predictions), threshold


class TikTokValueDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'][0],
            'attention_mask': encoding['attention_mask'][0],
            'labels': torch.tensor(label, dtype=torch.float)
        }

def train_with_cv(input_model_path, cur_val, texts, labels, splits, fold_metrics=None, num_epochs=5):
    """
    Modified train_with_cv to use pre-determined splits and track best performance
    """
    print(f"\nInitial data for {cur_val}:")
    print(f"Total samples: {len(labels)}")
    print(f"Positive labels (1): {sum(labels)}")
    print(f"Negative labels (0): {len(labels) - sum(labels)}")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    cv_results_train = []
    cv_results_test = []
    best_overall_train_metrics = {}
    best_overall_test_metrics = {}
    best_overall_fold_f1 = -1
    if(sum(labels)>30):

      for fold, (train_idx, val_idx) in enumerate(splits):
        print(f"\n{'-'*40}")
        print(f"Processing Fold {fold + 1}/{len(splits)}")
        print(f"{'-'*40}")

        # Split data using provided indices
        train_texts = [texts[i] for i in train_idx]
        train_labels = [labels[i] for i in train_idx]
        val_texts = [texts[i] for i in val_idx]
        val_labels = [labels[i] for i in val_idx]

        # Initialize model and tokenizer
        tokenizer = AutoTokenizer.from_pretrained(input_model_path)
        # model = RobertaForSequenceClassification.from_pretrained(
        model = AutoModelForSequenceClassification.from_pretrained(
            input_model_path,
            num_labels=1,
            problem_type="binary_classification").to(device)
            # ignore_mismatched_sizes=True           # XXXXXXXXXX


        # Create datasets
        train_dataset = TikTokValueDataset(train_texts, train_labels, tokenizer)
        val_dataset = TikTokValueDataset(val_texts, val_labels, tokenizer)

        # Create regular data loaders
        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=0
        )
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

        # Initialize optimizer and scheduler
        optimizer = AdamW(model.parameters(), lr=2e-5)
        num_training_steps = len(train_loader) * num_epochs
        num_warmup_steps = num_training_steps // 10
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=num_warmup_steps,
            num_training_steps=num_training_steps
        )

        # Initialize early stopping
        early_stopping = EarlyStopping(patience=2)

        best_val_f1 = 0
        best_model_state = None
        best_train_metrics = None
        best_val_metrics = None

        for epoch in range(num_epochs):
            print(f"\nEpoch {epoch + 1}/{num_epochs}")

            # Training
            model.train()
            train_loss = train_epoch(model, train_loader, optimizer, device, scheduler)
            print(f"Average training loss: {train_loss:.4f}")

            # Evaluation
            train_labels_pred, train_preds, train_threshold = evaluate_with_metrics(model, train_loader, device)
            train_metrics_dict = calculate_metrics_per_label(cur_val, train_labels_pred, train_preds)

            print("\nTraining Metrics:")
            print_detailed_metrics(train_metrics_dict)

            val_labels_pred, val_preds, val_threshold = evaluate_with_metrics(model, val_loader, device, threshold=train_threshold)
            val_metrics_dict = calculate_metrics_per_label(cur_val, val_labels_pred, val_preds)

            print("\nValidation Metrics:")
            print_detailed_metrics(val_metrics_dict)

            # Update best metrics
            if val_metrics_dict['f1'] > best_val_f1:
                best_val_f1 = val_metrics_dict['f1']
                best_model_state = model.state_dict().copy()
                best_train_metrics = train_metrics_dict
                best_val_metrics = val_metrics_dict

                # Update fold_metrics if this is the best performance seen
                if fold_metrics is not None and val_metrics_dict['f1'] > fold_metrics.get('best_f1', 0):
                    fold_metrics.update({
                        'best_f1': val_metrics_dict['f1'],
                        'best_fold': fold,
                        'best_split': (train_idx, val_idx)
                    })

            # Early stopping check
            early_stopping(val_metrics_dict['f1'])
            if early_stopping.early_stop:
                print("Early stopping triggered")
                break

        if best_val_f1 > best_overall_fold_f1:
            best_overall_fold_f1 = best_val_f1
            best_overall_train_metrics = best_train_metrics  # This already contains the best epoch's metrics
            best_overall_test_metrics = best_val_metrics

            # Update fold_metrics
            if fold_metrics is not None:
                fold_metrics.update({
                    'best_f1': best_val_f1,
                    'best_fold': fold,
                    'best_split': (train_idx, val_idx)
                })
        # Store best results from this fold
        cv_results_train.append(best_train_metrics)
        cv_results_test.append(best_val_metrics)

        # Clear GPU memory
        del model
        torch.cuda.empty_cache()
        gc.collect()


    else:

        best_overall_train_metrics  = {
          'label': cur_val,
          'precision': 0,
          'recall': 0,
          'f1': 0,
          'tp': 0,
          'fp': 0,
          'tn': 0,
          'fn': 0,
          'support': sum(labels)
        }
        best_overall_test_metrics  = {
          'label': cur_val,
          'precision': 0,
          'recall': 0,
          'f1': 0,
          'tp': 0,
          'fp': 0,
          'tn': 0,
          'fn': 0,
          'support': sum(labels)
        }

    return best_overall_train_metrics, best_overall_test_metrics

def print_detailed_metrics(metrics_dict):
    """Helper function to print detailed metrics"""
    print(f"True Positives: {metrics_dict['tp']}")
    print(f"True Negatives: {metrics_dict['tn']}")
    print(f"False Positives: {metrics_dict['fp']}")
    print(f"False Negatives: {metrics_dict['fn']}")
    print(f"Precision: {metrics_dict['precision']:.4f}")
    print(f"Recall: {metrics_dict['recall']:.4f}")
    print(f"F1 Score: {metrics_dict['f1']:.4f}")
    print(f"Support: {metrics_dict['support']}")



def print_detailed_metrics(metrics_dict):
    """Helper function to print detailed metrics"""
    print(f"True Positives: {metrics_dict['tp']}")
    print(f"True Negatives: {metrics_dict['tn']}")
    print(f"False Positives: {metrics_dict['fp']}")
    print(f"False Negatives: {metrics_dict['fn']}")
    print(f"Precision: {metrics_dict['precision']:.4f}")
    print(f"Recall: {metrics_dict['recall']:.4f}")
    print(f"F1 Score: {metrics_dict['f1']:.4f}")
    print(f"Support: {metrics_dict['support']}")


# Modified main execution
# First, filter long texts
all_texts = df['text'].tolist()
filtered_texts = []
filtered_indices = []  # Keep track of original indices
removed_count = 0

tokenizer = AutoTokenizer.from_pretrained(input_model_path)
for idx, text in enumerate(all_texts):
    tokens = tokenizer.encode(text, add_special_tokens=True)
    if len(tokens) <= 600:
        filtered_texts.append(text)
        filtered_indices.append(idx)
    else:
        removed_count += 1

print(f"Removed {removed_count} texts due to length limit")
# Create filtered dataframe
#df_filtered = df.iloc[filtered_indices]
df_filtered=df
# Initial train/test split
kf = KFold(n_splits=3, shuffle=True, random_state=42)
splits = list(kf.split(filtered_texts))

best_split_f1 = -1
best_split_idx = 0
best_train_idx = None
best_test_idx = None

# Process each label with the same splits
results_train = []
results_test = []
num_epochs = 3

for cur_val in df.columns[2:-2].tolist():
# for cur_val in df.columns[10:12].tolist():
    print(f"\nProcessing value: {cur_val}")
    df1 = df[['original_path', 'path', cur_val, 'text']].copy()

    # Print value distribution in original data
    print("\nValue distribution in original data:")
    print(df1[cur_val].value_counts())

    df1['label'] = df1[cur_val].astype(int)
    texts = df1['text'].tolist()
    labels = df1['label'].tolist()

    # Print sample of text and corresponding labels
    print("\nSample of 3 texts and their labels:")
    for i in range(min(3, len(texts))):
        print(f"\nText {i+1} (first 100 chars): {texts[i][:100]}...")
        print(f"Label: {labels[i]}")

    train_metrics, test_metrics = train_with_cv(input_model_path, cur_val, texts, labels, splits, fold_metrics=None, num_epochs=num_epochs)
    torch.cuda.empty_cache()#PyTorch is used to release unused GPU memory
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    results_train.append(train_metrics)
    results_test.append(test_metrics)

# Create DataFrames for final results
df_train_results = pd.DataFrame(results_train)
df_test_results = pd.DataFrame(results_test)

# Set index to label names
df_train_results.set_index('label', inplace=True)
df_test_results.set_index('label', inplace=True)

print("\nFinal Training Results:")
print(df_train_results)
print("\nFinal Test Results:")
print(df_test_results)

# Save results to CSV
training_res_file = "".join(['training_results',run_id,'.csv'])
df_train_results.to_csv(training_res_file)
files.download(training_res_file)

test_res_file = "".join(['test_results',run_id,'.csv'])
df_test_results.to_csv(test_res_file)
files.download(test_res_file)



# Save best split to GCP
#save_split_to_gcp(
#    best_train_idx,
#    best_test_idx,
#    project_id="iucc-social-media-influencers",
#    bucket_name="influencers_tiktok",
#    source_blob_name="split_train_test_best"
#)

Removed 158 texts due to length limit

Processing value: SDT

Value distribution in original data:
SDT
0    696
1    194
Name: count, dtype: int64

Sample of 3 texts and their labels:

Text 1 (first 100 chars): Genre: Comedy
Sound: Yes

ALEX: "Promise me you won't tell anyone."
FRIEND: "I promise!"
ALEX: "I st...
Label: 0

Text 2 (first 100 chars): Genre: Comedy
Sound: Yes

ALEX: "I mean, if you were gonna tell me that I was gonna be a pool boy fi...
Label: 0

Text 3 (first 100 chars): Genre: Comedy, with sound.

DAD: "Oh honey, look how cute our kids are."
MOM: *drinks from a mug*
[K...
Label: 0

Initial data for SDT:
Total samples: 890
Positive labels (1): 194
Negative labels (0): 696
Using device: cuda

----------------------------------------
Processing Fold 1/3
----------------------------------------


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at deberta-xlarge-pretrained-itamar-gcp and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(



Epoch 1/3


Training: 100%|██████████| 122/122 [01:39<00:00,  1.23it/s]



Prediction statistics:
Mean: 0.2123
Std: 0.1252
Min: 0.0329
Max: 0.6048
Average training loss: 0.5966


Evaluating: 100%|██████████| 122/122 [00:32<00:00,  3.70it/s]


... (1,255 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>